# 3.4 Survival Analysis

**Class 5:** among people with diabetes, **how long until they develop chronic kidney disease (CKD)**, and what's associated with getting it sooner?

1. Build a **time-to-event** table: one row per person with a start date, an end date, and whether CKD happened
2. **Kaplan–Meier** curves: the share still CKD-free over time
3. **Cox regression:** hazard ratios for baseline characteristics

Uses `survival_conditions.csv` and `last_visits.csv` (notebook 1.2), plus `person.csv` (1.1) and `measurements_wide.csv` (2.0).

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

In [ ]:
library(survival)
library(survminer)   # ggsurvplot()

survival_conditions <- load_data("survival_conditions.csv")
last_visits         <- load_data("last_visits.csv")
person              <- load_data("person.csv")
measurements_wide   <- load_data("measurements_wide.csv")

## Step 1: First diabetes and first CKD date per person

- **Time zero** (`diagnosis_date`): the first diabetes diagnosis
- **Event** (`ckd_date`): the first CKD diagnosis, if any

In [ ]:
first_dates <- survival_conditions %>%
  group_by(person_id) %>%
  summarise(
    diagnosis_date = suppressWarnings(min(condition_start_date[is_diabetes])),
    ckd_date       = suppressWarnings(min(condition_start_date[is_ckd]))
  ) %>%
  mutate(across(c(diagnosis_date, ckd_date), ~ if_else(is.infinite(.x), as.Date(NA), .x))) %>%
  filter(!is.na(diagnosis_date))

head(first_dates)

## Step 2: Time to event and censoring

- People whose CKD came **before or on** the diabetes diagnosis already had it: they're not at risk, so we **drop** them
- `status = 1` if CKD happened, `0` if not (**censored**)
- The end of follow-up is the CKD date, or the **last visit** for people without CKD
- `time_years` is the time from diagnosis to the end of follow-up

In [ ]:
surv_data <- first_dates %>%
  left_join(last_visits, by = "person_id") %>%
  filter(is.na(ckd_date) | ckd_date > diagnosis_date) %>%
  mutate(
    status     = if_else(!is.na(ckd_date), 1, 0),
    end_date   = if_else(status == 1, ckd_date, last_visit_date),
    time_years = as.numeric(end_date - diagnosis_date) / 365.25
  ) %>%
  filter(!is.na(time_years), time_years > 0)

surv_data %>% count(status)

## Step 3: Add baseline characteristics

Predictors must describe people **at time zero**, not things that happen later. (Using the CKD stage itself as a predictor of CKD would be circular.) We add:

- **age at diagnosis** and **gender** from `person.csv`
- **BMI** and **HbA1c** from `measurements_wide.csv`, and an HbA1c group (< 7% vs ≥ 7%)

> **Simplification:** these are each person's *most recent* BMI and HbA1c, not values measured at diagnosis. A real study would use the value closest to (and before) `diagnosis_date`.

In [ ]:
surv_data <- surv_data %>%
  left_join(person %>% select(person_id, date_of_birth, gender), by = "person_id") %>%
  left_join(measurements_wide %>% select(person_id, BMI, HbA1c), by = "person_id") %>%
  mutate(
    age_at_diagnosis = as.numeric(diagnosis_date - date_of_birth) / 365.25,
    hba1c_group = factor(if_else(HbA1c >= 7, "HbA1c >= 7%", "HbA1c < 7%"))
  )

glimpse(surv_data)

## Step 4: Kaplan–Meier curves

`Surv(time, status)` combines follow-up time and the event indicator. `survfit()` estimates the share still **CKD-free** over time; here we compare HbA1c groups. The **log-rank p-value** tests whether the curves differ.

> ⚠️ The **risk table** under the plot shows how many people are still followed. Counts of 1 to 20 (common at long follow-up times) **can't be shared**: don't screenshot it.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 9)

km_fit <- survfit(Surv(time_years, status) ~ hba1c_group, data = surv_data)

ggsurvplot(
  km_fit, data = surv_data,
  conf.int = TRUE, pval = TRUE, risk.table = TRUE,
  xlim = c(0, 10), break.time.by = 2,
  xlab = "Years since diabetes diagnosis",
  ylab = "Proportion CKD-free",
  ggtheme = theme_bw(base_size = 14)
)

**Interpret:** which group develops CKD sooner? Read off the share CKD-free at 5 years for each group.

## Step 5: Cox proportional-hazards model

`coxph()` estimates **hazard ratios (HR)**: how much each characteristic multiplies the instantaneous risk of CKD, holding the others constant.

- **HR > 1:** CKD sooner (higher risk); **HR < 1:** lower risk
- For age and BMI, the HR is per 1-unit increase

In [ ]:
cox_model <- coxph(Surv(time_years, status) ~ age_at_diagnosis + gender + BMI + hba1c_group,
                   data = surv_data)

broom::tidy(cox_model, exponentiate = TRUE, conf.int = TRUE) %>%
  select(term, estimate, conf.low, conf.high, p.value)

**Concordance (C-index)** is how well the model ranks people by risk: 0.5 is chance, 1 is perfect.

In [ ]:
summary(cox_model)$concordance

## Step 6: Check the proportional-hazards assumption

Cox models assume each HR stays constant over time. `cox.zph()` tests it: a small p-value for a variable suggests its effect changes over time.

In [ ]:
cox.zph(cox_model)

## Discuss

1. Write one HR in plain language: *"People with HbA1c ≥ 7% had X times the hazard of CKD compared with those below 7%, adjusting for age, gender, and BMI."*
2. Why must predictors be measured **at or before** time zero?
3. What other baseline factors might matter? (Blood pressure, medications, kidney function at diagnosis…)
4. Death competes with CKD as an outcome. How might that affect these results?